# RNA 3D Structure Prediction - Kaggle Inference Notebook

This notebook is optimized for creating Kaggle submissions from our trained model. It handles:

1. Loading the trained model
2. Processing test data in the Kaggle format
3. Running inference with the dual-mode approach (no dihedral angles at test time)
4. Formatting the predictions for Kaggle submission

In [ ]:
# Basic imports
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import math  # Needed for positional encoding
from pathlib import Path
from tqdm.notebook import tqdm

# Add project root to path for module imports
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Import project modules
from src.models.rna_folding_model import RNAFoldingModel
from src.data_loading import RNADataset, collate_fn, create_data_loader

# Apply data loading patch for multi-structure feature files
def fixed_load_precomputed_features(
    target_id, features_dir, temporal_cutoff=None
):
    """
    Enhanced version of load_precomputed_features that handles inconsistent feature file formats.
    """
    import os
    import numpy as np
    import warnings
    import pandas as pd
    
    features = {}

    # 1. Load dihedral features
    dihedral_path = os.path.join(
        features_dir, "dihedral_features", f"{target_id}_dihedral_features.npz"
    )
    if os.path.exists(dihedral_path):
        try:
            with np.load(dihedral_path) as data:
                # Check feature generation date if available for temporal cutoff
                if temporal_cutoff is not None and "metadata" in data:
                    try:
                        metadata_str = str(data["metadata"])
                        if "extraction_timestamp" in metadata_str:
                            timestamp_part = metadata_str.split("extraction_timestamp")[
                                1
                            ].split("'")[1]
                            generation_date = timestamp_part.split()[0]

                            if pd.to_datetime(generation_date) > pd.to_datetime(
                                temporal_cutoff
                            ):
                                warnings.warn(
                                    f"Dihedral features for {target_id} were generated after the temporal cutoff. Using zeros."
                                )
                                features["dihedral"] = None
                                return features
                    except (KeyError, IndexError, ValueError):
                        pass

                # Handle different feature file formats
                if "features" in data:
                    # Standard format
                    features["dihedral"] = {"features": data["features"].astype(np.float32)}
                elif "struct_1_features" in data:
                    # Multi-structure format with numbered structures
                    features["dihedral"] = {"features": data["struct_1_features"].astype(np.float32)}
                else:
                    # Unknown format - warn and use None
                    warnings.warn(f"Dihedral features file for {target_id} has unexpected format. Using zeros.")
                    features["dihedral"] = None
                    return features
                
                # Handle NaN values if present
                if features["dihedral"] is not None and np.isnan(features["dihedral"]["features"]).any():
                    features["dihedral"]["features"] = np.nan_to_num(
                        features["dihedral"]["features"], nan=0.0
                    )
        except Exception as e:
            # Handle any errors in loading
            warnings.warn(f"Error loading dihedral features for {target_id}: {str(e)}. Using zeros.")
            features["dihedral"] = None
    else:
        # For test data or if file is missing
        features["dihedral"] = None
        warnings.warn(f"Dihedral features not found for {target_id}. Using zeros.")

    # 2. Load thermodynamic features (required)
    thermo_path = os.path.join(
        features_dir, "thermo_features", f"{target_id}_thermo_features.npz"
    )
    if not os.path.exists(thermo_path):
        raise ValueError(
            f"Thermodynamic features not found for {target_id}. Required for prediction."
        )

    try:
        with np.load(thermo_path) as data:
            # Check feature generation date if available for temporal cutoff
            if temporal_cutoff is not None and "generation_date" in data:
                generation_date = str(data["generation_date"])
                if pd.to_datetime(generation_date) > pd.to_datetime(temporal_cutoff):
                    warnings.warn(
                        f"Thermo features for {target_id} were generated after the temporal cutoff. Using zeros."
                    )
                    features["thermo"] = None
                    return features

            # Extract key arrays and scalar values
            thermo_features = {}

            # Get pairing probabilities matrix (critical)
            if "pairing_probs" in data:
                thermo_features["pairing_probs"] = data["pairing_probs"].astype(np.float32)
            elif "base_pair_probs" in data:
                thermo_features["pairing_probs"] = data["base_pair_probs"].astype(np.float32)
            else:
                raise ValueError(f"No pairing probabilities found in {target_id} thermo features")

            # Handle NaN values in pairing probabilities
            if np.isnan(thermo_features["pairing_probs"]).any():
                thermo_features["pairing_probs"] = np.nan_to_num(
                    thermo_features["pairing_probs"], nan=0.0
                )

            # Get positional entropy (optional)
            if "positional_entropy" in data:
                thermo_features["positional_entropy"] = data["positional_entropy"].astype(
                    np.float32
                )
            else:
                # Calculate from pairing probabilities if missing
                pair_probs = thermo_features["pairing_probs"]
                row_entropies = -np.sum(
                    pair_probs * np.log2(pair_probs + 1e-10), axis=1
                )
                thermo_features["positional_entropy"] = row_entropies

            # Get accessibility (optional)
            if "accessibility" in data:
                thermo_features["accessibility"] = data["accessibility"].astype(np.float32)
            else:
                # Calculate from pairing probabilities if missing
                pair_probs = thermo_features["pairing_probs"]
                accessibilities = 1.0 - np.sum(pair_probs, axis=1)
                thermo_features["accessibility"] = np.maximum(0.0, accessibilities)

            features["thermo"] = thermo_features
    except Exception as e:
        raise ValueError(f"Error loading thermodynamic features for {target_id}: {str(e)}")

    # 3. Load evolutionary coupling features (optional)
    mi_path = os.path.join(
        features_dir, "evolutionary_features", f"{target_id}_evolutionary_features.npz"
    )
    if os.path.exists(mi_path):
        try:
            with np.load(mi_path) as data:
                # Check feature generation date if available for temporal cutoff
                if temporal_cutoff is not None and "generation_date" in data:
                    generation_date = str(data["generation_date"])
                    if pd.to_datetime(generation_date) > pd.to_datetime(temporal_cutoff):
                        warnings.warn(
                            f"Evolutionary features for {target_id} were generated after the temporal cutoff. Using zeros."
                        )
                        features["evolutionary"] = None
                        return features

                # Extract key arrays and metadata
                evol_features = {}

                # Get coupling matrix (required for this feature type)
                if "coupling_matrix" in data:
                    coupling_matrix = data["coupling_matrix"].astype(np.float32)
                    
                    # Check if the matrix is valid (not all zeros or constant)
                    is_valid = not np.allclose(coupling_matrix, 0.0)
                    evol_features["has_valid_mi"] = is_valid
                    
                    if is_valid:
                        evol_features["coupling_matrix"] = coupling_matrix
                    else:
                        # Zero matrix case - still provide the matrix but flag it
                        evol_features["coupling_matrix"] = coupling_matrix
                        warnings.warn(f"Coupling matrix for {target_id} is all zeros or constant.")
                else:
                    # No coupling matrix found
                    evol_features["has_valid_mi"] = False
                    evol_features["coupling_matrix"] = None

                features["evolutionary"] = evol_features
        except Exception as e:
            # Handle errors in evolutionary feature loading
            warnings.warn(f"Error loading evolutionary features for {target_id}: {str(e)}. Proceeding without them.")
            features["evolutionary"] = None
    else:
        # No evolutionary features available
        features["evolutionary"] = None

    return features

# Apply the patch
from src import data_loading
original_load_precomputed_features = data_loading.load_precomputed_features
data_loading.load_precomputed_features = fixed_load_precomputed_features
print("Successfully patched data_loading.load_precomputed_features with fixed version")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Configuration

Set paths and parameters for inference

In [ ]:
# Paths
# For local testing (these will change for Kaggle)
TEST_SEQUENCES_PATH = "../data/raw/test_sequences.csv"  # Update for Kaggle
FEATURES_DIR = "../data/processed/"  # Update for Kaggle
OUTPUT_DIR = "../submissions/"

# Model Checkpoint Options
# Add multiple checkpoint paths for model selection/ensembling
MODEL_PATHS = {
    "final_model": "../results/final_model/run_20250423-072601/checkpoints/best_model.pt",
    "tuning_lr_0.001": "../results/tuning_run_1/lr_0.001/run_20250423-072437/checkpoints/best_model.pt",
    "tuning_lr_0.0005": "../results/tuning_run_1/lr_0.0005/run_20250423-072448/checkpoints/best_model.pt",
    "tuning_lr_0.0001": "../results/tuning_run_1/lr_0.0001/run_20250423-072458/checkpoints/best_model.pt",
    "production_run_1": "../results/production_run_1/run_20250423-072209/checkpoints/best_model.pt"
}

# Choose which model to use (set to None to evaluate all and pick the best)
SELECTED_MODEL = None  # Options: None, or any key from MODEL_PATHS dictionary
USE_ENSEMBLE = False  # Set to True to use model ensembling

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameters
BATCH_SIZE = 8
NUM_SAMPLES = 5  # For Kaggle, we need 5 conformations per sequence
TEMPERATURE = 0.1  # Temperature for sampling diversity

# Set the May 2022 temporal cutoff date - critical for proper evaluation
TEMPORAL_CUTOFF = "2022-05-01"

## 2. Load Model

Load the trained model from checkpoint. 

**Note:** This code includes a fix for corrupted checkpoints that contain only a 'dummy' key in model_state_dict.
The fix initializes models from scratch using the correct architecture from the checkpoint, allowing
inference to proceed even when weights can't be loaded.

In [ ]:
def load_model(checkpoint_path):
    """Load model from checkpoint with robustness to corrupted state_dict."""
    print(f"Loading model from {checkpoint_path}")
    
    # Check if the file exists
    if not os.path.exists(checkpoint_path):
        print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
        return None, None, {'val_rmsd': None, 'epoch': None}
    
    try:
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Get model configuration
        if 'args' in checkpoint:
            config = checkpoint['args']
        elif 'model_config' in checkpoint:
            config = checkpoint['model_config']
        else:
            # Default configuration
            print("WARNING: No configuration found in checkpoint. Using defaults.")
            config = {
                'num_blocks': 4,
                'residue_embed_dim': 128,
                'pair_embed_dim': 64,
                'num_attention_heads': 4,
                'dropout': 0.1
            }
        
        # Initialize model
        model = RNAFoldingModel(config)
        
        # Check if state_dict is valid
        if 'model_state_dict' in checkpoint:
            model_state_dict = checkpoint['model_state_dict']
            # Check if it's corrupted (only contains dummy key)
            if list(model_state_dict.keys()) == ['dummy']:
                print("WARNING: Corrupted checkpoint detected with only 'dummy' key.")
                print("Initializing model from scratch with the configuration from checkpoint.")
                # We don't load weights - using freshly initialized weights
            else:
                # Load state dict if it seems valid
                model.load_state_dict(model_state_dict)
                print("Successfully loaded weights from checkpoint.")
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
            print("Successfully loaded weights from 'state_dict'.")
        else:
            print("WARNING: No state_dict found in checkpoint. Using untrained model.")
        
        # Move to device
        model = model.to(device)
        model.eval()
        
        # Extract validation metrics if available
        val_rmsd = None
        if 'val_rmsd' in checkpoint:
            val_rmsd = checkpoint['val_rmsd']
        elif 'validation_rmsd' in checkpoint:
            val_rmsd = checkpoint['validation_rmsd']
        elif 'best_val_metrics' in checkpoint and 'rmsd' in checkpoint['best_val_metrics']:
            val_rmsd = checkpoint['best_val_metrics']['rmsd']
        
        epoch = None
        if 'epoch' in checkpoint:
            epoch = checkpoint['epoch']
        
        # Enhance model to handle longer sequences
        patch_model_for_long_sequences(model)
        
        # Return model, config, and metrics
        return model, config, {'val_rmsd': val_rmsd, 'epoch': epoch}
    
    except Exception as e:
        print(f"ERROR loading model: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, {'val_rmsd': None, 'epoch': None}

# Enhanced positional encoding that can handle longer sequences
class EnhancedPositionalEncoding(nn.Module):
    """
    Enhanced version of PositionalEncoding that handles sequences longer than max_len.
    """
    
    def __init__(self, config):
        """Initialize positional encoding with extendable length."""
        super().__init__()

        # Extract parameters from config
        self.embed_dim = config.get("residue_embed_dim", 128)
        self.max_len = config.get("max_len", 500)

        # Create constant positional encoding matrix
        position = torch.arange(0, self.max_len).unsqueeze(1).float()
        self.div_term = torch.exp(
            torch.arange(0, self.embed_dim, 2).float() * (-math.log(10000.0) / self.embed_dim)
        )

        pe = torch.zeros(self.max_len, self.embed_dim)
        pe[:, 0::2] = torch.sin(position * self.div_term)
        pe[:, 1::2] = torch.cos(position * self.div_term)

        # Register buffer (not a parameter, but part of state)
        self.register_buffer("pe", pe.unsqueeze(0))  # Shape: (1, max_len, embed_dim)
    
    def extend_pe(self, new_max_len):
        """Dynamically extend the positional encoding to handle longer sequences."""
        # Create new positions
        old_max_len = self.pe.size(1)
        if new_max_len <= old_max_len:
            return  # No need to extend
            
        print(f"Extending positional encoding from {old_max_len} to {new_max_len}")
        
        # Generate positions for the new entries
        position = torch.arange(old_max_len, new_max_len).unsqueeze(1).float().to(self.pe.device)
        
        # Create new encodings
        pe_extension = torch.zeros(new_max_len - old_max_len, self.embed_dim, 
                                  device=self.pe.device)
        pe_extension[:, 0::2] = torch.sin(position * self.div_term.to(self.pe.device))
        pe_extension[:, 1::2] = torch.cos(position * self.div_term.to(self.pe.device))
        
        # Concatenate with existing buffer
        new_pe = torch.cat([self.pe.squeeze(0), pe_extension], dim=0).unsqueeze(0)
        
        # Replace the buffer
        self.pe = new_pe
        self.max_len = new_max_len
    
    def forward(self, seq_len):
        """Get positional encodings with automatic extension if needed."""
        if seq_len > self.max_len:
            # If sequence is longer than our current max, extend the encoding
            new_max_len = max(seq_len, int(self.max_len * 1.5))  # Grow by 50% to reduce frequent extensions
            self.extend_pe(new_max_len)
            
        return self.pe[:, :seq_len]

def patch_model_for_long_sequences(model):
    """Patch a model's positional encoding to handle long sequences."""
    if hasattr(model, 'embedding_module') and hasattr(model.embedding_module, 'positional_encoding'):
        # Get original module
        orig_pe = model.embedding_module.positional_encoding
        
        # Create config for new module
        config = {
            'residue_embed_dim': orig_pe.embed_dim,
            'max_len': orig_pe.max_len
        }
        
        # Create enhanced module
        enhanced_pe = EnhancedPositionalEncoding(config)
        
        # Copy the existing buffer
        enhanced_pe.pe = orig_pe.pe.clone()
        
        # Replace in the model
        model.embedding_module.positional_encoding = enhanced_pe
        
        print(f"Patched model with enhanced positional encoding (max_len: {enhanced_pe.max_len})")
        return True
    else:
        print("Warning: Could not find positional encoding in model structure")
        return False

def evaluate_models(model_paths, valid_loader=None):
    """Evaluate multiple models to select the best one.
    
    Args:
        model_paths: Dictionary of model paths
        valid_loader: Optional validation data loader
        
    Returns:
        Dictionary of loaded models with their metrics
    """
    models = {}
    
    # Load all models
    for name, path in model_paths.items():
        try:
            if os.path.exists(path):
                model, config, metrics = load_model(path)
                
                # Only add if model loaded successfully
                if model is not None:
                    models[name] = {
                        'model': model,
                        'config': config,
                        'metrics': metrics,
                        'path': path
                    }
                    
                    print(f"Loaded {name}:")
                    print(f"  Validation RMSD: {metrics['val_rmsd']}")
                    print(f"  Trained epochs: {metrics['epoch']}")
                    print()
            else:
                print(f"WARNING: Model path not found: {path}")
        except Exception as e:
            print(f"ERROR loading model {name}: {str(e)}")
    
    # If no validation loader, select the model with the best validation metrics
    if valid_loader is None:
        sorted_models = sorted(
            [(name, info) for name, info in models.items() if info['metrics']['val_rmsd'] is not None],
            key=lambda x: x[1]['metrics']['val_rmsd']
        )
        
        if sorted_models:
            best_name, best_info = sorted_models[0]
            print(f"\nBest model based on validation RMSD: {best_name}")
            print(f"Validation RMSD: {best_info['metrics']['val_rmsd']}")
            print(f"Trained epochs: {best_info['metrics']['epoch']}")
        else:
            print("\nWARNING: No models with validation metrics found")
    
    return models

# Load models
try:
    if SELECTED_MODEL is not None and SELECTED_MODEL in MODEL_PATHS:
        # Load only the selected model
        model_path = MODEL_PATHS[SELECTED_MODEL]
        model, config, metrics = load_model(model_path)
        
        if model is not None:
            models = {
                SELECTED_MODEL: {
                    'model': model,
                    'config': config,
                    'metrics': metrics,
                    'path': model_path
                }
            }
            print(f"Using selected model: {SELECTED_MODEL}")
        else:
            models = {}
            print(f"ERROR: Failed to load selected model: {SELECTED_MODEL}")
    else:
        # Evaluate all models
        models = evaluate_models(MODEL_PATHS)
        
        # Choose the best model if not using ensemble
        if not USE_ENSEMBLE:
            # Sort by validation RMSD
            sorted_models = sorted(
                [(name, info) for name, info in models.items() if info['metrics']['val_rmsd'] is not None],
                key=lambda x: x[1]['metrics']['val_rmsd']
            )
            
            if sorted_models:
                best_name, best_info = sorted_models[0]
                # Keep only the best model
                models = {best_name: best_info}
                print(f"Selected best model: {best_name}")
            else:
                print("WARNING: No models with validation metrics. Using the first available model.")
                if models:
                    first_name = list(models.keys())[0]
                    models = {first_name: models[first_name]}
                    print(f"Using model: {first_name}")
                else:
                    # Handle the case when no models loaded successfully
                    print("ERROR: No valid models were loaded.")
                    models = {}
    
    print(f"Using {len(models)} model(s) for inference")
    
except Exception as e:
    print(f"ERROR during model loading: {str(e)}")
    import traceback
    traceback.print_exc()
    models = {}

## 3. Load Test Data

Create data loader for test sequences

In [ ]:
def create_test_loader(sequences_path, features_dir, batch_size=8):
    """Create test data loader."""
    # Check if files exist
    if not os.path.exists(sequences_path):
        print(f"ERROR: Test sequences file not found at {sequences_path}")
        return None
    
    if not os.path.exists(features_dir):
        print(f"ERROR: Features directory not found at {features_dir}")
        return None
    
    try:
        # Create dataset
        dataset = RNADataset(
            sequences_csv_path=sequences_path,
            features_dir=features_dir,
            temporal_cutoff=TEMPORAL_CUTOFF,  # This is CRITICAL for proper competition constraints
            use_validation_set=True,  # For test time, this flag makes sure we use the restricted feature set
            require_features=False,  # Allow sequences without all features
        )
        
        # Create data loader
        loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=4,
        )
        
        print(f"Test loader created with {len(dataset)} sequences")
        return loader
    except Exception as e:
        print(f"ERROR creating test loader: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Create test loader
try:
    test_loader = create_test_loader(TEST_SEQUENCES_PATH, FEATURES_DIR, BATCH_SIZE)
    
    if test_loader is None:
        print("WARNING: Failed to create test loader. Inference cannot continue.")
except Exception as e:
    print(f"ERROR: {str(e)}")
    import traceback
    traceback.print_exc()
    test_loader = None

## 4. Run Inference

Run inference on test data with multiple samples per sequence

In [ ]:
def generate_samples(model, batch, num_samples=5, temperature=0.1):
    """Generate multiple structure samples for each sequence in the batch.
    
    Args:
        model: RNA folding model
        batch: Input batch
        num_samples: Number of structure samples to generate
        temperature: Temperature for sampling diversity
        
    Returns:
        List of dictionaries containing samples
    """
    results = []
    
    # Get batch device and size
    batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                   for k, v in batch.items()}
    
    # Generate samples
    for i in range(num_samples):
        # Forward pass with noise scale controlled by temperature
        with torch.no_grad():
            # Set dropout for diverse sampling
            model.train()  # Enable dropout for diverse sampling
            outputs = model(batch_device)
            model.eval()  # Restore eval mode
            
            # Collect results
            for j, target_id in enumerate(batch['target_ids']):
                seq_len = batch['lengths'][j].item()
                
                # Get coordinates and confidence
                coords = outputs['pred_coords'][j, :seq_len].cpu().numpy()
                conf = torch.sigmoid(outputs['pred_confidence'][j, :seq_len]).cpu().numpy()
                
                # Add to results
                results.append({
                    'target_id': target_id,
                    'sample_id': i,
                    'coords': coords,
                    'confidence': conf,
                })
    
    return results

def run_inference(model, data_loader, num_samples=5, temperature=0.1):
    """Run inference on all test data.
    
    Args:
        model: RNA folding model
        data_loader: Test data loader
        num_samples: Number of structure samples to generate per sequence
        temperature: Temperature for sampling diversity
        
    Returns:
        Dictionary mapping target_id to list of sample dictionaries
    """
    all_results = {}
    
    # Process all batches
    for batch in tqdm(data_loader, desc="Running inference"):
        # Generate samples
        samples = generate_samples(model, batch, num_samples, temperature)
        
        # Organize by target_id
        for sample in samples:
            target_id = sample['target_id']
            if target_id not in all_results:
                all_results[target_id] = []
            all_results[target_id].append(sample)
    
    return all_results

In [ ]:
# Define ensemble inference function
def run_ensemble_inference(models_dict, data_loader, num_samples=5, temperature=0.1):
    """Run inference using an ensemble of models.
    
    Args:
        models_dict: Dictionary of models
        data_loader: Test data loader
        num_samples: Number of structure samples to generate per sequence
        temperature: Temperature for sampling diversity
        
    Returns:
        Dictionary mapping target_id to list of sample dictionaries
    """
    all_results = {}
    model_weights = {}
    
    # Check if we have any models to work with
    if not models_dict:
        print("ERROR: No models available for inference")
        return all_results
    
    # Determine weights based on validation RMSD (lower RMSD = higher weight)
    valid_models = [(name, info) for name, info in models_dict.items() 
                   if info['metrics']['val_rmsd'] is not None]
    
    if valid_models:
        # Invert RMSD to get weights (lower RMSD = higher weight)
        rmsd_values = [info['metrics']['val_rmsd'] for _, info in valid_models]
        min_rmsd = min(rmsd_values)
        
        # Calculate weights (normalized)
        for name, info in valid_models:
            rmsd = info['metrics']['val_rmsd']
            # Use inverse squared RMSD for more pronounced weighting
            model_weights[name] = (min_rmsd / rmsd) ** 2
        
        # Normalize weights
        weight_sum = sum(model_weights.values())
        if weight_sum > 0:
            for name in model_weights:
                model_weights[name] /= weight_sum
    else:
        # Equal weights if no validation metrics
        for name in models_dict:
            model_weights[name] = 1.0 / len(models_dict)
    
    print("Model ensemble weights:")
    for name, weight in model_weights.items():
        print(f"  {name}: {weight:.4f}")
    
    # For each batch in the dataloader
    for batch in tqdm(data_loader, desc="Running ensemble inference"):
        # Get batch device
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                       for k, v in batch.items()}
        
        # Generate samples from each model
        all_model_samples = {}
        
        # First pass: collect predictions from all models
        for name, model_info in models_dict.items():
            model = model_info['model']
            weight = model_weights.get(name, 0.0)
            
            # Skip model if weight is too low (optional optimization)
            if weight < 0.05:
                continue
                
            # Generate samples for this model
            model_samples = generate_samples(model, batch_device, num_samples, temperature)
            
            # Organize by target_id and sample_id
            for sample in model_samples:
                target_id = sample['target_id']
                sample_id = sample['sample_id']
                
                if target_id not in all_model_samples:
                    all_model_samples[target_id] = {}
                
                if sample_id not in all_model_samples[target_id]:
                    all_model_samples[target_id][sample_id] = []
                
                # Store sample with its weight
                all_model_samples[target_id][sample_id].append({
                    'coords': sample['coords'],
                    'confidence': sample['confidence'],
                    'weight': weight
                })
        
        # Second pass: combine predictions with weighted averaging
        for target_id, samples_dict in all_model_samples.items():
            if target_id not in all_results:
                all_results[target_id] = []
            
            for sample_id, model_preds in samples_dict.items():
                # Get first prediction to determine shape
                first_pred = model_preds[0]
                combined_coords = np.zeros_like(first_pred['coords'])
                combined_conf = np.zeros_like(first_pred['confidence'])
                total_weight = 0.0
                
                # Weighted average of coordinates and confidence
                for pred in model_preds:
                    weight = pred['weight']
                    combined_coords += pred['coords'] * weight
                    combined_conf += pred['confidence'] * weight
                    total_weight += weight
                
                # Normalize by total weight
                if total_weight > 0:
                    combined_coords /= total_weight
                    combined_conf /= total_weight
                
                # Add combined prediction to results
                all_results[target_id].append({
                    'target_id': target_id,
                    'sample_id': sample_id,
                    'coords': combined_coords,
                    'confidence': combined_conf
                })
    
    return all_results

# Run inference - either with a single model or ensemble
if USE_ENSEMBLE and len(models) > 1:
    print("Running ensemble inference with multiple models...")
    results = run_ensemble_inference(models, test_loader, NUM_SAMPLES, TEMPERATURE)
else:
    # Check if we have any models
    if not models:
        print("ERROR: No models available for inference. Check model paths and loading.")
        results = {}  # Return empty results to avoid errors in subsequent cells
    else:
        # Get the first (and only) model from the dictionary
        model_name = list(models.keys())[0]
        model = models[model_name]['model']
        print(f"Running inference with single model: {model_name}")
        results = run_inference(model, test_loader, NUM_SAMPLES, TEMPERATURE)

## 5. Format for Kaggle Submission

Format the results according to Kaggle submission requirements

In [ ]:
def format_kaggle_submission(results):
    """Format results for Kaggle submission.
    
    Args:
        results: Dictionary mapping target_id to list of sample dictionaries
        
    Returns:
        Pandas DataFrame formatted according to Kaggle submission guidelines
    """
    # Check if results are empty
    if not results:
        print("WARNING: No results to format for submission")
        return pd.DataFrame(columns=['target_id', 'model_id', 'coordinates', 'confidence'])
    
    # Prepare submission data
    submission_rows = []
    
    for target_id, samples in results.items():
        # Must have exactly 5 samples per target
        if len(samples) < 5:
            print(f"WARNING: Target {target_id} has only {len(samples)} samples. Need 5.")
            # Duplicate the last sample if needed
            if samples:  # Make sure we have at least one sample to duplicate
                samples = samples + [samples[-1]] * (5 - len(samples))
            else:
                print(f"ERROR: No samples for target {target_id}")
                continue
        elif len(samples) > 5:
            # Keep only the first 5
            samples = samples[:5]
        
        # Create rows for each sample
        for i, sample in enumerate(samples):
            try:
                # Format coordinates as string
                coords_str = json.dumps(sample['coords'].tolist())
                confidence_str = json.dumps(sample['confidence'].tolist())
                
                # Add to submission rows
                submission_rows.append({
                    'target_id': target_id,
                    'model_id': i,
                    'coordinates': coords_str,
                    'confidence': confidence_str,
                })
            except Exception as e:
                print(f"ERROR processing sample {i} for target {target_id}: {str(e)}")
                print(f"Sample keys: {list(sample.keys())}")
    
    # Create DataFrame
    submission_df = pd.DataFrame(submission_rows)
    
    # Verify we have data
    if len(submission_df) == 0:
        print("WARNING: Empty submission DataFrame created")
    
    return submission_df

# Format results for submission
submission_df = format_kaggle_submission(results)
print(f"Submission DataFrame created with {len(submission_df)} rows")

# Display preview
if not submission_df.empty:
    display(submission_df.head())
else:
    print("No submission data to display")

In [ ]:
# Generate a meaningful filename for the submission
if USE_ENSEMBLE and len(models) > 1:
    model_identifier = "ensemble"
else:
    model_identifier = list(models.keys())[0] if models else "no_model"
    
# Add timestamp to ensure uniqueness
timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
submission_filename = f"submission_{model_identifier}_{timestamp}.csv"
submission_path = os.path.join(OUTPUT_DIR, submission_filename)

# Check if we have data to save
if submission_df.empty:
    print("WARNING: Empty submission DataFrame. No file will be saved.")
else:
    # Save submission file
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    submission_df.to_csv(submission_path, index=False)
    print(f"Submission saved to {submission_path}")

# Also create a submission metadata file with model information
metadata = {
    "timestamp": timestamp,
    "models_used": list(models.keys()) if models else [],
    "ensemble": USE_ENSEMBLE,
    "temperature": TEMPERATURE,
    "num_samples": NUM_SAMPLES,
    "metrics": {name: info["metrics"] for name, info in models.items()} if models else {},
    "submission_size": len(submission_df),
    "num_targets": len(submission_df['target_id'].unique()) if not submission_df.empty else 0
}

# Save metadata even if submission is empty (for debugging)
os.makedirs(OUTPUT_DIR, exist_ok=True)
metadata_path = os.path.join(OUTPUT_DIR, f"metadata_{model_identifier}_{timestamp}.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Submission metadata saved to {metadata_path}")

## 6. Submission Verification

Verify that the submission file meets Kaggle's requirements

In [ ]:
def verify_submission(submission_df):
    """Verify that the submission meets Kaggle requirements."""
    # Check if DataFrame is empty
    if submission_df.empty:
        print("ERROR: Submission DataFrame is empty")
        return False
    
    # Check required columns
    required_columns = ['target_id', 'model_id', 'coordinates', 'confidence']
    for col in required_columns:
        if col not in submission_df.columns:
            print(f"ERROR: Missing required column '{col}'")
            return False
    
    # Check target_id and model_id combinations
    target_ids = submission_df['target_id'].unique()
    for target_id in target_ids:
        target_samples = submission_df[submission_df['target_id'] == target_id]
        if len(target_samples) != 5:
            print(f"ERROR: Target {target_id} has {len(target_samples)} samples. Need exactly 5.")
            return False
        
        model_ids = sorted(target_samples['model_id'].values)
        if model_ids != [0, 1, 2, 3, 4]:
            print(f"ERROR: Target {target_id} has invalid model_ids: {model_ids}. Need [0,1,2,3,4].")
            return False
        
        # Check coordinate and confidence formats
        for _, row in target_samples.iterrows():
            try:
                coords = json.loads(row['coordinates'])
                conf = json.loads(row['confidence'])
                
                if len(coords) == 0 or len(conf) == 0:
                    print(f"ERROR: Empty coordinates or confidence for {target_id}, model {row['model_id']}")
                    return False
                
                if len(coords) != len(conf):
                    print(f"ERROR: Coordinate and confidence length mismatch for {target_id}, model {row['model_id']}")
                    return False
                
                # Check coordinate format
                if not all(isinstance(c, list) and len(c) == 3 for c in coords):
                    print(f"ERROR: Invalid coordinate format for {target_id}, model {row['model_id']}")
                    return False
                
                # Check confidence is between 0 and 1
                if not all(0 <= c <= 1 for c in conf):
                    print(f"ERROR: Confidence values outside [0,1] for {target_id}, model {row['model_id']}")
                    return False
                
            except json.JSONDecodeError:
                print(f"ERROR: Invalid JSON format for {target_id}, model {row['model_id']}")
                return False
            except Exception as e:
                print(f"ERROR checking {target_id}, model {row['model_id']}: {str(e)}")
                return False
    
    print("✅ Submission verification passed!")
    return True

# Verify submission if we have data
if not submission_df.empty:
    verification_result = verify_submission(submission_df)
    print(f"Verification result: {'Success' if verification_result else 'Failed'}")
else:
    print("No submission data to verify")

## 7. Visualization

Visualize a few predicted structures

def visualize_prediction(target_id, model_id=0):
    """Visualize a predicted structure."""
    try:
        import py3Dmol
    except ImportError:
        print("py3Dmol not available. Unable to visualize.")
        return
    
    # Check if we have a submission
    if submission_df.empty:
        print("No submission data to visualize.")
        return
    
    # Check if the target_id and model_id exist
    matching_rows = submission_df[(submission_df['target_id'] == target_id) & 
                        (submission_df['model_id'] == model_id)]
    
    if matching_rows.empty:
        print(f"No data found for target_id={target_id}, model_id={model_id}")
        return
    
    try:
        # Get the prediction
        row = matching_rows.iloc[0]
        
        coords = np.array(json.loads(row['coordinates']))
        confidence = np.array(json.loads(row['confidence']))
        
        # Create visualization
        view = py3Dmol.view(width=800, height=600)
        
        # Add atoms with confidence-based coloring
        for i, (coord, conf) in enumerate(zip(coords, confidence)):
            # Color based on confidence (red-white-blue gradient)
            r = max(0, min(1, 2*(1-conf)))
            b = max(0, min(1, 2*conf-1))
            g = max(0, min(1, 1-2*abs(conf-0.5)))
            color = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
            
            # Add atom as sphere
            view.addSphere({
                'center': {'x': coord[0], 'y': coord[1], 'z': coord[2]},
                'radius': 0.4,
                'color': color,
                'opacity': 0.85
            })
            
            # Add bonds (connect consecutive atoms)
            if i > 0:
                view.addCylinder({
                    'start': {'x': coords[i-1][0], 'y': coords[i-1][1], 'z': coords[i-1][2]},
                    'end': {'x': coord[0], 'y': coord[1], 'z': coord[2]},
                    'radius': 0.1,
                    'color': 'gray',
                    'opacity': 0.7
                })
        
        # Set view options
        view.zoomTo()
        view.setStyle({'stick': {}})
        
        # Show title
        print(f"Prediction for {target_id} (Model {model_id})")
        print(f"Number of residues: {len(coords)}")
        print(f"Average confidence: {confidence.mean():.4f}")
        
        return view
    except Exception as e:
        print(f"Error visualizing prediction: {str(e)}")
        return None

# Choose a sequence to visualize if we have data
if not submission_df.empty:
    # Find first target_id
    try:
        example_target_id = submission_df['target_id'].iloc[0]
        print(f"Visualizing target: {example_target_id}")
        view = visualize_prediction(example_target_id, model_id=0)
        if view is not None:
            view.show()
    except Exception as e:
        print(f"Error setting up visualization: {str(e)}")
else:
    print("No submission data available for visualization")

In [ ]:
def generate_execution_log():
    """
    Generate a comprehensive execution log from notebook outputs.
    
    This function captures all cell outputs from the Jupyter kernel,
    including errors, and creates a markdown report for debugging.
    """
    from IPython import get_ipython
    from datetime import datetime
    import os
    import json
    import sys
    import traceback
    
    # Get notebook shell
    shell = get_ipython()
    
    # Get all cells and their outputs
    cells = list(shell.history_manager.get_range())
    
    # Generate timestamp
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    
    # Create markdown report
    report = [
        f"# Kaggle Inference Notebook Execution Log",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        "## Environment Information",
        f"Device used: {device}",
        f"Python version: {sys.version}",
        f"PyTorch version: {torch.__version__}",
        "",
        "## Configuration",
        "```python",
    ]
    
    # Add configuration details
    configs = {
        "MODEL_PATHS": MODEL_PATHS,
        "SELECTED_MODEL": SELECTED_MODEL,
        "USE_ENSEMBLE": USE_ENSEMBLE,
        "BATCH_SIZE": BATCH_SIZE,
        "NUM_SAMPLES": NUM_SAMPLES,
        "TEMPERATURE": TEMPERATURE
    }
    
    report.append(json.dumps(configs, indent=2))
    report.append("```")
    report.append("")
    
    # Add cell outputs
    report.append("## Cell Execution Results")
    
    # Keep track of cell execution count
    cell_count = 1
    
    # Get the user namespace to execute code and retrieve variables
    user_ns = shell.user_ns
    
    # Simplified approach to get error information
    error_dict = {}
    
    # Iterate through executed cells
    for session, line, exec_count in cells:
        # Skip empty cells and the log generator itself
        line = line.strip()
        if not line or "generate_execution_log" in line:
            continue
            
        # Add cell input
        report.append(f"### Cell {cell_count}: Input")
        report.append("```python")
        report.append(line)
        report.append("```")
        
        # Try to capture the output
        report.append(f"### Cell {cell_count}: Output")
        report.append("```")
        
        # Check if variables were assigned in this cell by inspecting the code
        var_names = []
        if "=" in line and not line.lstrip().startswith(("if", "for", "while", "def", "class")):
            # Simple variable assignment detection
            parts = line.split("=")[0].strip().split()
            if parts:
                var_names.append(parts[-1])
        
        # For each potential variable, try to grab its value from user namespace
        for var_name in var_names:
            if var_name in user_ns:
                try:
                    var_value = user_ns[var_name]
                    # For DataFrames, show a preview
                    if 'pandas.core.frame.DataFrame' in str(type(var_value)):
                        report.append(f"{var_name} (shape: {var_value.shape}):")
                        report.append(str(var_value.head()))
                    # For dictionaries with simple values, show content
                    elif isinstance(var_value, dict) and len(var_value) < 10:
                        report.append(f"{var_name} (dict with {len(var_value)} items):")
                        report.append(str(var_value))
                    # For lists with reasonable size
                    elif isinstance(var_value, list) and len(var_value) < 10:
                        report.append(f"{var_name} (list with {len(var_value)} items):")
                        report.append(str(var_value))
                    else:
                        report.append(f"{var_name}: {type(var_value).__name__} created")
                except Exception as e:
                    report.append(f"Error retrieving value for {var_name}: {str(e)}")
                    
        report.append("```")
        report.append("")
        
        cell_count += 1
    
    # Add a section for errors with line numbers from the running session
    report.append("## Errors Detected")
    
    # Try to get the current IPython instance's error history
    ip = get_ipython()
    
    if hasattr(ip, '_last_traceback'):
        report.append("Most recent error traceback:")
        report.append("```")
        last_error = "".join(traceback.format_tb(ip._last_traceback))
        report.append(last_error)
        report.append("```")
    else:
        report.append("No recent error traceback available.")
    
    # Add model information
    report.append("## Model Information")
    if 'models' in user_ns:
        model_info = user_ns['models']
        report.append(f"Number of models loaded: {len(model_info)}")
        for model_name, info in model_info.items():
            report.append(f"### {model_name}")
            if 'metrics' in info:
                report.append("**Metrics:**")
                for metric_name, metric_value in info['metrics'].items():
                    report.append(f"- {metric_name}: {metric_value}")
            report.append("")
    else:
        report.append("No models loaded or model information not available.")
    
    # Add specific debugging for cell 10 and 11 (model loading and inference)
    report.append("## Model Loading/Inference Debug Info")
    if 'models' in user_ns:
        report.append(f"models dictionary has {len(user_ns['models'])} entries.")
    else:
        report.append("models variable not found.")
    
    if 'results' in user_ns:
        report.append(f"results dictionary has {len(user_ns['results'])} entries.")
        
        # Sample one result entry if available
        if len(user_ns['results']) > 0:
            first_key = list(user_ns['results'].keys())[0]
            first_samples = user_ns['results'][first_key]
            report.append(f"First target ({first_key}) has {len(first_samples)} samples.")
            
            if len(first_samples) > 0:
                # Check for expected fields in sample
                first_sample = first_samples[0]
                report.append(f"First sample keys: {list(first_sample.keys())}")
                
                # Check shapes for coords and confidence
                if 'coords' in first_sample:
                    report.append(f"coords shape: {first_sample['coords'].shape}")
                if 'confidence' in first_sample:
                    report.append(f"confidence shape: {first_sample['confidence'].shape}")
    else:
        report.append("results variable not found.")
    
    # Add submission information
    report.append("## Submission Information")
    if 'submission_df' in user_ns:
        sub_df = user_ns['submission_df']
        report.append(f"Submission shape: {sub_df.shape}")
        report.append(f"Target IDs: {len(sub_df['target_id'].unique())}")
        report.append(f"Model IDs per target: {sub_df.groupby('target_id').size().mean()}")
        report.append("")
        report.append("**Sample entries:**")
        report.append("```")
        report.append(str(sub_df.head(3)))
        report.append("```")
    else:
        report.append("No submission DataFrame found.")
    
    # Add verification results
    report.append("## Verification Results")
    if 'verify_submission' in user_ns:
        report.append("Verification function is available.")
        if 'submission_df' in user_ns:
            # Don't actually call verify_submission as it might have side effects
            report.append("Run verification manually with verify_submission(submission_df).")
    else:
        report.append("Verification function not found.")
    
    # Save the report
    report_text = "\n".join(report)
    
    # Create reports directory if it doesn't exist
    os.makedirs("../notebook_reports", exist_ok=True)
    
    # Save report
    report_path = f"../notebook_reports/kaggle_inference_log_{timestamp}.md"
    with open(report_path, "w") as f:
        f.write(report_text)
    
    print(f"Execution log saved to {report_path}")
    return report_path

try:
    # Import traceback directly here to ensure it's available
    import traceback
    
    # Generate the execution log
    execution_log_path = generate_execution_log()
    print(f"Log generation complete! Report saved to {execution_log_path}")
except Exception as e:
    print(f"Error generating log: {str(e)}")
    import traceback  # Import again just to be super safe
    traceback.print_exc()

In [ ]:
def visualize_prediction(target_id, model_id=0):
    """Visualize a predicted structure."""
    try:
        import py3Dmol
    except ImportError:
        print("py3Dmol not available. Unable to visualize.")
        return
    
    # Get the prediction
    row = submission_df[(submission_df['target_id'] == target_id) & 
                        (submission_df['model_id'] == model_id)].iloc[0]
    
    coords = np.array(json.loads(row['coordinates']))
    confidence = np.array(json.loads(row['confidence']))
    
    # Create visualization
    view = py3Dmol.view(width=800, height=600)
    
    # Add atoms with confidence-based coloring
    for i, (coord, conf) in enumerate(zip(coords, confidence)):
        # Color based on confidence (red-white-blue gradient)
        r = max(0, min(1, 2*(1-conf)))
        b = max(0, min(1, 2*conf-1))
        g = max(0, min(1, 1-2*abs(conf-0.5)))
        color = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
        
        # Add atom as sphere
        view.addSphere({
            'center': {'x': coord[0], 'y': coord[1], 'z': coord[2]},
            'radius': 0.4,
            'color': color,
            'opacity': 0.85
        })
        
        # Add bonds (connect consecutive atoms)
        if i > 0:
            view.addCylinder({
                'start': {'x': coords[i-1][0], 'y': coords[i-1][1], 'z': coords[i-1][2]},
                'end': {'x': coord[0], 'y': coord[1], 'z': coord[2]},
                'radius': 0.1,
                'color': 'gray',
                'opacity': 0.7
            })
    
    # Set view options
    view.zoomTo()
    view.setStyle({'stick': {}})
    
    # Show title
    print(f"Prediction for {target_id} (Model {model_id})")
    print(f"Number of residues: {len(coords)}")
    print(f"Average confidence: {confidence.mean():.4f}")
    
    return view

# Choose a sequence to visualize
if len(submission_df) > 0:
    example_target_id = submission_df['target_id'].iloc[0]
    view = visualize_prediction(example_target_id, model_id=0)
    if view is not None:
        view.show()